In [8]:
import pandas as pd
import numpy as np
import joblib
import os
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import classification_report, make_scorer, f1_score
from sklearn.calibration import CalibratedClassifierCV

In [9]:
import warnings

warnings.filterwarnings('ignore')

In [10]:
full_data = pd.read_csv('../data/processed/features_engineered.csv')

targets = ['Sleep_Quality_Num', 'Stress_Level_Num', 'Health_Issues_Num']

# Drop targets to get features
X = full_data.drop(columns=targets + ['Cluster', 'UMAP1', 'UMAP2'], errors='ignore')
Y = full_data[targets]

In [11]:
preprocessor = joblib.load('../models/preprocessor.joblib')

In [12]:
def run_cross_validation(X_data, y_data, target_name):
    print(f"\n--- Running 5-Fold CV for: {target_name} ---")
    
    # We use StratifiedKFold to keep the balance of classes consistent
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    model = LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1)
    
    # We transform X through the preprocessor before CV
    X_processed = preprocessor.transform(X_data)
    
    cv_results = cross_validate(
        model, X_processed, y_data, 
        cv=skf, 
        scoring=['accuracy', 'f1_macro'],
        return_train_score=False
    )
    
    print(f"Mean Accuracy: {cv_results['test_accuracy'].mean():.4f} (+/- {cv_results['test_accuracy'].std() * 2:.4f})")
    print(f"Mean F1 Macro: {cv_results['test_f1_macro'].mean():.4f}")
    return cv_results

In [14]:
cv_summary = {}
for target in targets:
    cv_summary[target] = run_cross_validation(X, Y[target], target)


--- Running 5-Fold CV for: Sleep_Quality_Num ---
Mean Accuracy: 0.9695 (+/- 0.0059)
Mean F1 Macro: 0.9642

--- Running 5-Fold CV for: Stress_Level_Num ---
Mean Accuracy: 0.9784 (+/- 0.0044)
Mean F1 Macro: 0.9670

--- Running 5-Fold CV for: Health_Issues_Num ---
Mean Accuracy: 0.9837 (+/- 0.0030)
Mean F1 Macro: 0.9771


In [15]:
def tune_and_calibrate(X_data, y_data, target_name):
    print(f"\n--- Tuning & Calibrating: {target_name} ---")
    
    X_processed = preprocessor.transform(X_data)
    
    # Define the Search Space
    param_dist = {
        'num_leaves': [20, 31, 50, 70],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [100, 200, 500],
        'max_depth': [-1, 10, 20],
        'subsample': [0.8, 0.9, 1.0]
    }
    
    lgbm = LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1)
    
    # Randomized Search
    search = RandomizedSearchCV(
        lgbm, param_distributions=param_dist, 
        n_iter=10, cv=3, scoring='f1_macro', random_state=42
    )
    search.fit(X_processed, y_data)
    
    best_model = search.best_estimator_
    print(f"Best Params: {search.best_params_}")
    
    # --- Step 4: Calibration ---
    # This makes the probability outputs more reliable for health predictions
    calibrated_model = CalibratedClassifierCV(best_model, method='sigmoid', cv=3)
    calibrated_model.fit(X_processed, y_data)
    
    # Save the Refined Model
    os.makedirs('../models/refined', exist_ok=True)
    save_path = f'../models/refined/lgbm_refined_{target_name}.joblib'
    joblib.dump(calibrated_model, save_path)
    print(f"Refined model saved to: {save_path}")
    
    return calibrated_model

In [16]:
refined_models = {}
for target in targets:
    refined_models[target] = tune_and_calibrate(X, Y[target], target)


--- Tuning & Calibrating: Sleep_Quality_Num ---
Best Params: {'subsample': 0.8, 'num_leaves': 20, 'n_estimators': 500, 'max_depth': -1, 'learning_rate': 0.05}
Refined model saved to: ../models/refined/lgbm_refined_Sleep_Quality_Num.joblib

--- Tuning & Calibrating: Stress_Level_Num ---
Best Params: {'subsample': 0.8, 'num_leaves': 20, 'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.05}
Refined model saved to: ../models/refined/lgbm_refined_Stress_Level_Num.joblib

--- Tuning & Calibrating: Health_Issues_Num ---
Best Params: {'subsample': 0.8, 'num_leaves': 20, 'n_estimators': 100, 'max_depth': -1, 'learning_rate': 0.05}
Refined model saved to: ../models/refined/lgbm_refined_Health_Issues_Num.joblib
